# Neobank onboarding funnel — exploratory data analysis

**Business question:** the growth team rolled out a new, lighter onboarding flow to half of new users.
Did it increase signup completion — and does that lift hold up once we look at the rest of the funnel,
segments, and time-to-activate?

This notebook works directly off the local CSVs (`data/raw_users.csv`, `data/raw_events.csv`), generated by
`scripts/generate_data.py`. It is the exploratory companion to:
- `analysis/ab_test.py` — the formal two-proportion z-test, CI, and power calculation
- `analysis/sample_size.py` — the a-priori sample size this experiment should have been designed with
- `dashboard/build_dashboard.py` — an interactive HTML version of the charts below
- `ml/train_activation_model.py` — a predictive model for post-signup activation

Run `python scripts/generate_data.py` first if `data/*.csv` don't exist yet.

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FUNNEL = ["signup_started", "signup_completed", "kyc_submitted", "first_deposit", "first_transaction"]

## 1. Load and sanity-check the data

In [ ]:
users = pd.read_csv("../data/raw_users.csv", parse_dates=["signup_ts"])
events = pd.read_csv("../data/raw_events.csv", parse_dates=["event_ts"])

print(f"users:  {len(users):,} rows, {users['user_id'].nunique():,} unique")
print(f"events: {len(events):,} rows")
print(f"missing values -- users: {users.isnull().sum().sum()}, events: {events.isnull().sum().sum()}")
users.head()

In [ ]:
events.head()

In [ ]:
# basic randomisation check: are the arms balanced, and is assignment independent of geography/platform?
print(users["variant"].value_counts(normalize=True))
print()
print(pd.crosstab(users["country"], users["variant"], normalize="index").round(3))

## 2. The funnel, overall

`signup_started -> signup_completed -> kyc_submitted -> first_deposit -> first_transaction`.
Each step is conditional on the previous one.

In [ ]:
rows = []
for step in FUNNEL:
    reached = set(events.loc[events.event_name == step, "user_id"])
    is_reached = users["user_id"].isin(reached)
    converted = int(is_reached.sum())
    rows.append({"step": step, "users": converted, "cumulative_rate": converted / len(users)})

funnel_overall = pd.DataFrame(rows)
funnel_overall["step_conversion"] = funnel_overall["users"] / funnel_overall["users"].shift(1)
funnel_overall

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(funnel_overall["step"], funnel_overall["cumulative_rate"] * 100, color="#2E86C1")
for i, row in funnel_overall.iterrows():
    ax.text(i, row["cumulative_rate"] * 100 + 1, f"{row['cumulative_rate']:.1%}", ha="center")
ax.set_ylabel("% of users reaching this step")
ax.set_title("Overall funnel — cumulative conversion")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 3. The experiment: does the new onboarding flow lift `signup_completed`?

Two-proportion z-test, matching `analysis/ab_test.py` exactly (kept here so the reasoning is visible
step by step, rather than hidden behind a CLI call).

In [ ]:
reached_completed = set(events.loc[events.event_name == "signup_completed", "user_id"])
users["reached_signup_completed"] = users["user_id"].isin(reached_completed)

g = users.groupby("variant")["reached_signup_completed"].agg(["sum", "count"])
c_conv, c_tot = int(g.loc["control", "sum"]), int(g.loc["control", "count"])
t_conv, t_tot = int(g.loc["treatment", "sum"]), int(g.loc["treatment", "count"])

p_c, p_t = c_conv / c_tot, t_conv / t_tot
p_pool = (c_conv + t_conv) / (c_tot + t_tot)
se_pool = math.sqrt(p_pool * (1 - p_pool) * (1 / c_tot + 1 / t_tot))
z = (p_t - p_c) / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

se_diff = math.sqrt(p_c * (1 - p_c) / c_tot + p_t * (1 - p_t) / t_tot)
diff = p_t - p_c
ci_low, ci_high = diff - 1.96 * se_diff, diff + 1.96 * se_diff

print(f"control   : {p_c:.2%}  ({c_conv:,}/{c_tot:,})")
print(f"treatment : {p_t:.2%}  ({t_conv:,}/{t_tot:,})")
print(f"absolute lift : {diff:+.2%}   95% CI [{ci_low:+.2%}, {ci_high:+.2%}]")
print(f"z = {z:.3f}   p = {p_value:.5f}")
print("\nSHIP" if (p_value < 0.05 and ci_low > 0) else "\nDO NOT SHIP")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["control", "treatment"], [p_c * 100, p_t * 100], color=["#7f8c8d", "#2E86C1"])
ax.set_ylabel("signup_completed rate (%)")
ax.set_title(f"+{diff:.1%} absolute lift  (p = {p_value:.4f})")
plt.tight_layout()
plt.show()

## 4. Does the lift propagate through the rest of the funnel?

The variant only changes `signup_completed` directly (see `scripts/generate_data.py`), but because every
later step is conditional on the one before it, more people *entering* the funnel downstream means more
people exiting it at the end too -- even though the per-step conversion *rates* downstream are not
variant-driven.

In [ ]:
rows = []
for step in FUNNEL:
    reached = set(events.loc[events.event_name == step, "user_id"])
    is_reached = users["user_id"].isin(reached)
    for variant, sub in users.groupby("variant"):
        converted = int(is_reached[sub.index].sum())
        total = len(sub)
        rows.append({"step": step, "variant": variant, "converted": converted, "total": total})

funnel = pd.DataFrame(rows)
funnel["rate"] = funnel["converted"] / funnel["total"]

lift_rows = []
for step in FUNNEL:
    sub = funnel[funnel.step == step]
    c = sub[sub.variant == "control"].iloc[0]
    t = sub[sub.variant == "treatment"].iloc[0]
    se = math.sqrt(c.rate * (1 - c.rate) / c.total + t.rate * (1 - t.rate) / t.total)
    d = t.rate - c.rate
    lift_rows.append({"step": step, "lift_pp": d * 100, "ci_half_pp": 1.96 * se * 100})

lift = pd.DataFrame(lift_rows)
lift

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lift["step"], lift["lift_pp"], yerr=lift["ci_half_pp"], capsize=4, color="#2E86C1")
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("lift, treatment - control (pp)")
ax.set_title("A/B lift at every funnel step, with 95% CI")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**Reading this chart:** the lift is largest at `signup_completed` (the directly-affected step, ~+5.9pp)
and *shrinks* at each step after that. That is expected and correct: a constant *relative* improvement in
who enters the next stage produces a shrinking *absolute* gap as conversion compounds downward. It does
not mean the effect is "wearing off" -- the relative lift on `first_transaction` is still close to the
original ~9-10%. Worth saying explicitly if asked about this chart.

## 5. Segment cuts: where does activation differ, and why?

Unlike the variant effect, country and platform genuinely do affect the downstream steps in this dataset
(see the `*_MODIFIER` constants in `scripts/generate_data.py`) -- e.g. ID-verification friction by country,
deposit propensity by platform. This is exactly the kind of cut a product analyst would pull to find where
to focus.

In [ ]:
activated_users = set(events.loc[events.event_name == "first_transaction", "user_id"])
users["activated"] = users["user_id"].isin(activated_users)

heat = users.pivot_table(index="country", columns="platform", values="activated", aggfunc="mean") * 100

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="Blues", ax=ax, cbar_kws={"label": "% activated"})
ax.set_title("Activation rate (first_transaction) by country x platform")
plt.tight_layout()
plt.show()

**Reading this:** AE consistently lags other markets (harder KYC friction in this synthetic story), and
`web` lags `ios`/`android` everywhere. In a real review this is where I'd propose two concrete follow-ups:
a KYC-flow audit for AE, and a web-onboarding UX review -- both testable as their own A/B tests, sized with
`analysis/sample_size.py` before launch.

## 6. Time to activate

In [ ]:
start = events.loc[events.event_name == "signup_started", ["user_id", "event_ts"]].rename(columns={"event_ts": "t0"})
txn = events.loc[events.event_name == "first_transaction", ["user_id", "event_ts"]].rename(columns={"event_ts": "t1"})
tta = start.merge(txn, on="user_id").merge(users[["user_id", "variant"]], on="user_id")
tta["minutes_to_activate"] = (tta["t1"] - tta["t0"]).dt.total_seconds() / 60

fig, ax = plt.subplots(figsize=(8, 4))
for variant, color in [("control", "#7f8c8d"), ("treatment", "#2E86C1")]:
    sub = tta[tta.variant == variant]
    ax.hist(sub["minutes_to_activate"], bins=40, alpha=0.6, label=variant, color=color)
ax.set_xlabel("minutes from signup_started to first_transaction")
ax.set_ylabel("users")
ax.legend()
ax.set_title("Time to activate, control vs treatment")
plt.tight_layout()
plt.show()

tta.groupby("variant")["minutes_to_activate"].describe()[["count", "mean", "50%", "std"]]

**Reading this:** the two distributions overlap almost completely -- once a user reaches
`signup_completed`, how fast they move through the rest of the funnel is unrelated to which onboarding
flow they saw. That is consistent with the experiment design: the treatment only changes whether someone
completes signup, not their behaviour afterwards.

## 7. Key takeaways

- **Ship the new onboarding flow.** +5.9pp absolute / ~+9.5% relative lift on `signup_completed`,
  p < 0.001, 95% CI excludes 0, and the effect propagates positively through the entire funnel to
  `first_transaction`.
- **The lift is real and isolated.** It shows up exactly where it should (`signup_completed`) and nowhere
  else (time-to-activate is unaffected) -- a useful sanity check that the experiment isn't picking up some
  unrelated confound.
- **Country (AE) and platform (web) are the two clearest places to focus next**, independent of the
  experiment -- each is a candidate for its own sized, isolated A/B test.
- **Next steps in this repo:** `analysis/sample_size.py` for designing the AE/web follow-up tests,
  `ml/train_activation_model.py` for a model that scores post-signup drop-off risk for targeting, and
  `dashboard/build_dashboard.py` for a shareable, interactive version of the charts above.